In [5]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import datetime

from memory.short_term_memory import ShortTermMemory
from memory.long_term_memory import LongTermMemory

from tools.facts_retrieve import retrieve_facts_schema
from function_call import select_service
from prompts import SYSTEM_PROMPT

load_dotenv()
client = OpenAI(
    base_url="https://freellmapi-seyc.onrender.com/v1",
    api_key=os.environ.get("FREE_LLM_API")
)


stm = ShortTermMemory(mode ="summary_buffer", llm_client=client)
ltm = LongTermMemory(llm_client=client)


session_id = ltm.session_id

In [ ]:
user_input = input("Input your query: ")
messages = [{"role": "system", "content": SYSTEM_PROMPT}] + stm.slide_chat + [{"role": "user", "content": user_input}]
while True:
    if user_input.strip().lower() == "exit":
        ltm.add(stm.slide_chat)
        break

    print("\n\n"+str(messages)+"\n\n")
    response = client.chat.completions.create(
        model="auto",
        messages=messages,
        tools=[retrieve_facts_schema],
        tool_choice='auto'
    )
    # print(response.choices[0].message)

    if response.choices[0].message.content != '' or response.choices[0].message.content != None:
        print(response.choices[0].message.content)
        stm.add(user_msg=user_input, assistant_msg=response.choices[0].message.content)

    if response.choices[0].message.tool_calls:
        stm.add(user_msg=user_input, assistant_msg=response.choices[0].message.tool_calls)

        for tool_call in response.choices[0].message.tool_calls:
            tool_result = select_service(tool_call.function)
            print(tool_result)
            stm.add(assistant_msg={"role": "tool", "tool_call_id":tool_call.id, "content": str(tool_result)}, role="tool")
            messages = [{"role": "system", "content": SYSTEM_PROMPT}] + stm.slide_chat

    if response.choices[0].finish_reason == "stop":
        user_input = input("Input your query: ")
        messages = [{"role": "system", "content": SYSTEM_PROMPT}] + stm.slide_chat + [{"role": "user", "content": user_input}]



[{'role': 'system', 'content': 'You are a helpful AI assistant with access to memory about this user, built from past conversations.\n\nYou have one memory tools:\n\n- **retrieve_facts**: retrieves distilled facts about the user (preferences, identity, constraints — e.g. "user prefers concise answers", "user works with Python"). Use this when the user\'s current request could be informed by something you may already know about them, their preferences, or their working context.\n\n## When to search memory\n\nSearch BEFORE answering, not after, when:\n- The user asks about their own preferences, past decisions, or history with you\n- The user references something implicitly ("the usual approach", "like before")\n- Answering well requires knowing something specific about this user that a generic answer wouldn\'t capture\n\nDo NOT search memory for:\n- Generic factual/technical questions unrelated to the user\'s personal context\n- Simple greetings or small talk\n\n## After retrieving\n\

KeyboardInterrupt: 

In [ ]:
stm.raw_chat


[{'role': 'user', 'content': 'HI'},
 {'role': 'assistant', 'content': 'Hello! How can I help you today?'},
 {'role': 'user', 'content': 'whats my name.'},
 {'role': 'assistant',
  'content': [ChatCompletionMessageFunctionToolCall(id='fc_108468ea-fbba-4c69-beab-5d9808d5c265', function=Function(arguments='{"query":"user name","top_k":5}', name='retrieve_facts'), type='function')]},
 {'role': 'tool',
  'tool_call_id': 'fc_108468ea-fbba-4c69-beab-5d9808d5c265',
  'content': '["user\'s name is satyam sharma", \'user loves dogs\', \'user likes their knowledge and beliefs to be challenged\', \'user prefers logic over emotions\']'},
 {'role': 'user', 'content': 'whats my name.'},
 {'role': 'assistant',
  'content': [ChatCompletionMessageFunctionToolCall(id='fc_52af49cf-8a96-4371-b775-8d3fd0651720', function=Function(arguments='{"query":"user name","top_k":5}', name='retrieve_facts'), type='function')]},
 {'role': 'tool',
  'tool_call_id': 'fc_52af49cf-8a96-4371-b775-8d3fd0651720',
  'content':